In [ ]:
"""
TRL GRPO Training for Sokoban — mirrors MLX GRPO notebook structure.
Model: LiquidAI/LFM2-1.2B-Tool

Structure follows the MLX notebook exactly:
  1. Config
  2. Load puzzles
  3. Build training dataset
  4. Build validation dataset
  5. Train
  6. Plot results
  7. Save final model
"""

import numpy as np
import matplotlib.pyplot as plt

from core.microban import load_microban
from rl import build_dataset
from rl.grpo import GRPOConfig, train_grpo, debug_generation, save_merged_model

print("=" * 60)
print("TRL GRPO for Sokoban  —  LFM2-1.2B-Tool")
print("=" * 60)


# ============================================================================
# 1. Configuration
# ============================================================================

config = GRPOConfig(
    model_id="LiquidAI/LFM2-1.2B-Tool",

    # Data
    repr_key="13_ACTION_CENTRIC",
    num_train_puzzles=50,
    max_steps_per_puzzle=30,

    # GRPO (matching MLX defaults)
    num_generations=4,          # MLX group_size=4
    learning_rate=1e-5,         # MLX learning_rate=1e-5
    beta=0.02,                  # MLX beta=0.02
    epsilon=0.2,                # MLX epsilon=0.2
    max_completion_length=265,  # MLX max_response_len=265
    num_epochs=3.0,
    per_device_batch_size=2,    # MLX batch_size=2
    gradient_accumulation_steps=4,

    # LoRA (matching MLX)
    lora_r=8,                   # MLX lora_rank=8
    lora_alpha=10,              # MLX lora_scale=10
    lora_dropout=0.0,           # MLX lora_dropout=0.0

    # LFM2 is small — no need for 4-bit on most hardware; flip to True if OOM
    load_in_4bit=False,

    output_dir="./rl-output",
    logging_steps=10,
    save_steps=100,
    seed=42,
)


# ============================================================================
# 2. Load puzzles
# ============================================================================

puzzles = load_microban("Microban.txt")
train_puzzles = [p for p in puzzles if p.num_boxes <= 3][: config.num_train_puzzles]
val_puzzles = [p for p in puzzles if p.num_boxes <= 3][
    config.num_train_puzzles : config.num_train_puzzles + 10
]

print(f"\nTraining puzzles:   {len(train_puzzles)}")
print(f"Validation puzzles: {len(val_puzzles)}")


# ============================================================================
# 3. Build training dataset
# ============================================================================

print("\nBuilding training dataset...")
hf_train_dataset = build_dataset(
    puzzles=train_puzzles,
    repr_key=config.repr_key,           # from GRPOConfig — mirrors MLX config.repr_key
    max_steps_per_puzzle=config.max_steps_per_puzzle,
    verbose=True,
)

print(f"  Total training examples: {len(hf_train_dataset)}")

# Sanity-check the first example (mirrors MLX notebook check)
sample = hf_train_dataset[0]
print("=" * 60)
print("SAMPLE PROMPT (first 500 chars):")
print("=" * 60)
print(sample["prompt"][:500])
print("...")
print("\nDoes it contain the board?")
print("  '####' in prompt:", "####" in sample["prompt"])
print("  '@'    in prompt:", "@" in sample["prompt"])
print("  '$'    in prompt:", "$" in sample["prompt"])
print(f"\n  optimal_move:  {sample.get('optimal_move')}")
print(f"  optimal_cost:  {sample.get('optimal_cost')}")


# ============================================================================
# 4. Build validation dataset
# ============================================================================

print("\nBuilding validation dataset...")
hf_val_dataset = build_dataset(
    puzzles=val_puzzles,
    repr_key=config.repr_key,
    max_steps_per_puzzle=config.max_steps_per_puzzle,
    verbose=True,
)

print(f"  Total validation examples: {len(hf_val_dataset)}")


# ============================================================================
# 5. Debug generation before training (mirrors MLX notebook)
# ============================================================================

print("\n" + "=" * 60)
print("Starting GRPO Training")
print(f"  Model:        {config.model_id}")
print(f"  Iterations:   {config.num_epochs} epochs")
print(f"  Group size:   {config.num_generations}")
print(f"  Batch size:   {config.per_device_batch_size}")
print(f"  Learning rate:{config.learning_rate}")
print(f"  beta (KL):    {config.beta}")
print(f"  epsilon:      {config.epsilon}")
print("=" * 60 + "\n")

# Load model temporarily for pre-training debug generation.
# train_grpo will reload it internally — this just mirrors what the
# MLX notebook does before calling grpo_train_loop.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

_tokenizer = AutoTokenizer.from_pretrained(config.model_id)
if _tokenizer.pad_token_id is None:
    _tokenizer.pad_token_id = _tokenizer.eos_token_id

_model = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
_model.eval()

debug_generation(_model, _tokenizer, sample["prompt"])

# Free memory before training loads its own copy
del _model
torch.cuda.empty_cache()


# ============================================================================
# 6. Train
# ============================================================================

trainer = train_grpo(
    dataset=hf_train_dataset,
    config=config,
    eval_dataset=hf_val_dataset,
)


# ============================================================================
# 7. Plot results  (mirrors MLX notebook plot)
# ============================================================================

log_history = trainer.state.log_history

# Extract loss and reward from TRL log history
losses  = [e["loss"]          for e in log_history if "loss"   in e]
rewards = [e["train/reward"]  for e in log_history if "train/reward" in e]

if losses and rewards:
    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.set_xlabel("Step")
    ax1.set_ylabel("Loss", color="tab:red")
    ax1.plot(losses, color="tab:red", alpha=0.7)
    ax1.tick_params(axis="y", labelcolor="tab:red")

    window = min(20, len(losses))
    smoothed_loss = np.convolve(losses, np.ones(window) / window, mode="valid")
    ax1.plot(
        range(window - 1, len(losses)),
        smoothed_loss,
        color="darkred",
        linewidth=2,
        label="Smoothed loss",
    )

    ax2 = ax1.twinx()
    ax2.set_ylabel("Reward", color="tab:blue")
    n = min(20, len(rewards))
    moving_avg = np.convolve(rewards, np.ones(n) / n, mode="valid")
    ax2.plot(
        range(n - 1, len(rewards)),
        moving_avg,
        color="tab:blue",
        linewidth=2,
        label="Reward (MA)",
    )
    ax2.tick_params(axis="y", labelcolor="tab:blue")

    plt.title("GRPO Training: Loss vs Reward")
    fig.tight_layout()

    import os
    os.makedirs(config.output_dir, exist_ok=True)
    curve_path = os.path.join(config.output_dir, "training_curves.png")
    plt.savefig(curve_path, dpi=150)
    plt.show()
    print(f"\n✓ Training curves saved to {curve_path}")
else:
    print("\n⚠ No loss/reward entries found in log history — skipping plot.")


# ============================================================================
# 8. Save final model
# ============================================================================

adapter_path = os.path.join(config.output_dir, "adapters")
os.makedirs(adapter_path, exist_ok=True)

# Save LoRA adapters (light, shareable)
trainer.model.save_pretrained(adapter_path)
trainer.processing_class.save_pretrained(adapter_path)
print(f"✓ LoRA adapters saved to {adapter_path}")

# Optionally merge weights into a standalone model
merged_path = os.path.join(config.output_dir, "merged")
save_merged_model(trainer, merged_path)